In [ ]:
# !pip install datasets
# !pip install transformers[torch]
# !pip install tf-keras

In [ ]:
import pandas as pd
from helper_func import clean_text
from helper_func import custom_train_validation_split

# Load data

df_train = pd.read_csv("/home/jack/github/kaggle/scoring/data/train.csv")


df_train.head()


In [ ]:
df_train = clean_text(df_train, 'full_text')
df_train.head()

In [ ]:
df_train, df_val = custom_train_validation_split(df_train, 0.3, random_state=42)

In [ ]:
df_train.reset_index(drop=True, inplace=True)

In [ ]:
df_val.reset_index(drop=True, inplace=True)

In [ ]:
# Adjust and rename labels to ensure no duplication
train_df = df_train[['essay_id', 'clean_text', 'score']].copy()
val_df = df_val[['essay_id', 'clean_text', 'score']].copy()

# Ensure labels are set correctly and avoid duplicates
train_df['labels'] = train_df['score'] - 1
val_df['labels'] = val_df['score'] - 1

In [ ]:
# import pandas as pd
# from datasets import Dataset
# from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, Trainer, TrainingArguments
# import gc  # For garbage collection
# import requests

# # Set a longer timeout (e.g., 30 seconds)
# requests.adapters.DEFAULT_RETRIES = 5  # Retry if there's a transient issue
# timeout = (10, 30)  # Connection and read timeout (in seconds)

# # Initialize tokenizer and model with custom timeout
# tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-large", timeout=timeout)
# model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-large", num_labels=6)




# # Tokenization function without re-adding labels
# def tokenize(sample):
#     return tokenizer(
#         sample['clean_text'], 
#         max_length=1024, 
#         truncation=True, 
#         padding='max_length'  # Consistent length
#     )

# # Create datasets and remove unneeded columns, ensuring no duplicated labels
# train_dataset = Dataset.from_pandas(train_df).map(tokenize)
# # train_dataset = train_dataset.add_column("labels", train_df['labels'])  # Add 'labels' explicitly
# train_dataset = train_dataset.remove_columns(['essay_id', 'clean_text'])

# val_dataset = Dataset.from_pandas(val_df).map(tokenize)
# # val_dataset = val_dataset.add_column("labels", val_df['labels'])
# val_dataset = val_dataset.remove_columns(['essay_id', 'clean_text'])

# # Set the format for Trainer
# train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
# val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# # Define training arguments with smaller batch size to avoid memory issues
# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,  # Smaller batch size to reduce memory load
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="./logs",  # Enable logging
#     logging_steps=100,     # Log every 100 steps
# )

# # Initialize the Trainer with correct datasets
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     data_collator=DataCollatorWithPadding(tokenizer),
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
# )

# # Train the model with try-except block to handle potential errors
# try:
#     trainer.train()
# except Exception as e:
#     print("Training failed:", e)

# # Explicitly trigger garbage collection to reduce memory
# gc.collect()


In [ ]:
# # Load tokenizer (assumes you have a tokenizer in one location for all models)
# tokenizer_path = "/path/to/tokenizer"  # Adjust to your local path
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

# # Load models for each fold
# model_paths = [
#     "/path/to/fold_0",
#     "/path/to/fold_1",
#     "/path/to/fold_2",
#     "/path/to/fold_3",
#     "/path/to/fold_4"
# ]

# models = [AutoModelForSequenceClassification.from_pretrained(path) for path in model_paths]

# # Setup training arguments
# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="./logs",
#     logging_steps=100,
# )

# # Placeholder for cross-validation results
# oof_predictions = []

# # Example of cross-validation
# # Here, you might use different fold datasets or a different setup for cross-validation
# # This loop shows how you could save the predictions for each fold
# for fold_number in range(5):  # Assuming 5 folds
#     # Create a new trainer for this fold
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         data_collator=DataCollatorWithPadding(tokenizer),
#         train_dataset=train_dataset,
#         eval_dataset=val_dataset,
#     )
#     # Train the model
#     try:
#         trainer.train()
#     except Exception as e:
#         print("Training failed:", e)  

#     # After training, make predictions on the validation set for this fold
#     # Here, you would need to define the validation dataset for each fold
#     validation_results = trainer.predict(train_dataset)  # Adjust to your validation set

#     # Store the predictions and labels in a dictionary or DataFrame
#     oof_fold_data = {
#         "fold": fold_number,
#         "predictions": validation_results.predictions,
#         "true_labels": validation_results.label_ids
#     }
#     oof_predictions.append(oof_fold_data)

# # After all folds, save the OOF data to a pickle file
# oof_path = "./oof.pkl"

# with open(oof_path, 'wb') as f:
#     pickle.dump(oof_predictions, f)

#     # Train the model
     



In [ ]:
# import pandas as pd

# # Load the OOF data from a pickle file
# oof_path = "/path/to/oof.pkl"
# oof_data = pd.read_pickle(oof_path)  # Use appropriate path


In [ ]:
# # Predict and retain the identifier for merging

# predictions = trainer.predict(tokenized_dataset['val'])
# pred_labels = predictions.predictions.argmax(axis=1)

# # Create a DataFrame with predictions and the unique identifier
# pred_df = pd.DataFrame({
#     'essay_id': val_df['essay_id'],
#     'predicted_label': pred_labels,
#     'true_label': val_df['labels']
# })

# # Now you can merge this prediction DataFrame with other data as needed

In [ ]:
# Save the model after training the pretrained deberta model

# model.save_pretrained("deberta_model")

In [ ]:
import pandas as pd 
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)
from datasets import Dataset
from glob import glob
import gc
import torch
from scipy.special import softmax

MAX_LENGTH = 1024
# TEST_DATA_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
MODEL_PATH = '/home/jack/github/kaggle/scoring/deberta/*/*'


EVAL_BATCH_SIZE = 10          # lower if having memory issues


models = glob(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(models[0])

def tokenize(sample):
    return tokenizer(sample['clean_text'], max_length=MAX_LENGTH, truncation=True)

ds = Dataset.from_pandas(train_df).map(tokenize).remove_columns(['essay_id', 'clean_text'])

args = TrainingArguments(
    ".", 
    per_device_eval_batch_size=EVAL_BATCH_SIZE, 
    report_to="none"
)

predictions = []
# Wrap the loop with tqdm to track progress
for model_path in tqdm(models, desc="Processing Models"):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    trainer = Trainer(
        model=model, 
        args=args, 
        data_collator=DataCollatorWithPadding(tokenizer), 
        tokenizer=tokenizer
    )
    
    preds = trainer.predict(ds).predictions
    predictions.append(softmax(preds, axis=-1))  # Apply softmax to get probabilities
    del model, trainer  # Free memory
    torch.cuda.empty_cache()
    gc.collect()  # Collect garbage to reduce memory usage
    
# After processing, calculate the average prediction score
predicted_score = 0.

# Wrap this loop with tqdm to track progress
for p in tqdm(predictions, desc="Averaging Predictions"):
    predicted_score += p
    
predicted_score /= len(predictions)

In [ ]:
predicted_score

In [ ]:
# Add the probabilities as new columns to the training dataframe.
for i in range(predicted_score.shape[1]):
    train_df[f'deberta_prob_{i}'] = predicted_score[:, i]

# Display the updated dataframe
train_df.drop(columns=['labels'], inplace=True)
train_df.head()

In [ ]:
train_df.to_csv("deberta_train_predictions.csv", index=False)

In [ ]:
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    DataCollatorWithPadding
)
from datasets import Dataset
from glob import glob
import gc
import torch
from scipy.special import softmax

MAX_LENGTH = 1024
# TEST_DATA_PATH = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
MODEL_PATH = '/home/jack/github/kaggle/scoring/deberta/*/*'


EVAL_BATCH_SIZE = 10          # lower if having memory issues


models = glob(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(models[0])

def tokenize(sample):
    return tokenizer(sample['clean_text'], max_length=MAX_LENGTH, truncation=True)
    

ds = Dataset.from_pandas(val_df).map(tokenize).remove_columns(['essay_id', 'clean_text'])

args = TrainingArguments(
    ".", 
    per_device_eval_batch_size=EVAL_BATCH_SIZE, 
    report_to="none"
)

predictions = []
# Wrap the loop with tqdm to track progress
for model_path in tqdm(models, desc="Processing Models"):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    trainer = Trainer(
        model=model, 
        args=args, 
        data_collator=DataCollatorWithPadding(tokenizer), 
        tokenizer=tokenizer
    )
    
    preds = trainer.predict(ds).predictions
    predictions.append(softmax(preds, axis=-1))  # Apply softmax to get probabilities
    del model, trainer  # Free memory
    torch.cuda.empty_cache()
    gc.collect()  # Collect garbage to reduce memory usage
    
# After processing, calculate the average prediction score
predicted_score = 0.

# Wrap this loop with tqdm to track progress
for p in tqdm(predictions, desc="Averaging Predictions"):
    predicted_score += p
    
predicted_score /= len(predictions)

In [ ]:
# Add the probabilities as new columns to the training dataframe.
for i in range(predicted_score.shape[1]):
    val_df[f'deberta_prob_{i}'] = predicted_score[:, i]

# Display the updated dataframe
val_df.drop(columns=['labels'], inplace=True)
val_df.head()

In [ ]:
val_df.tail()

In [ ]:
val_df.to_csv("deberta_val_predictions.csv", index=False)


In [ ]:
ful_df = pd.concat([train_df, val_df], axis=0)

In [ ]:
len(ful_df)

In [ ]:
ful_df.head()

In [ ]:
ful_df.to_csv("deberta_full_predictions.csv", index=False)  

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer

# Sample text data

# Initialize the HashingVectorizer
hashing_vectorizer = HashingVectorizer(n_features=200, alternate_sign=False)

# Transform the text data into hash vector features
hashed_features = hashing_vectorizer.transform(texts)

# Convert to dense representation for easier manipulation (not recommended for large datasets)
hashed_features_dense = hashed_features.todense()

# Display the hash vector features
print("Hash vector features:\n", hashed_features_dense)


In [ ]:

# Import necessary libraries
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
import torch
import gc

# Model and tokenizer paths for XLNet
XLNET_MODEL_PATH = 'xlnet-base-cased'  # Adjust to your desired XLNet variant

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get the number of unique labels in the dataset (assuming 'train_df' is defined)
unique_labels = set(train_df['labels'])  # Update as necessary
num_classes = len(unique_labels)

# Create configuration with the correct number of labels
xlnet_config = AutoConfig.from_pretrained(XLNET_MODEL_PATH, num_labels=num_classes)

# Initialize tokenizer and model with correct configuration
xlnet_tokenizer = AutoTokenizer.from_pretrained(XLNET_MODEL_PATH)
xlnet_model = AutoModelForSequenceClassification.from_config(xlnet_config).to(device)

# Tokenize text data for XLNet
def tokenize_xlnet(sample):
    return xlnet_tokenizer(sample['clean_text'], max_length=512, truncation=True)

# Load dataset for XLNet
xlnet_ds = Dataset.from_pandas(train_df).map(tokenize_xlnet).remove_columns(['essay_id', 'clean_text'])

# Set training arguments for XLNet
xlnet_args = TrainingArguments(
    "xlnet_predictions",
    per_device_eval_batch_size=10,
    report_to="none"
)

# Create a trainer for XLNet
xlnet_trainer = Trainer(
    model=xlnet_model,
    args=xlnet_args,
    data_collator=DataCollatorWithPadding(xlnet_tokenizer),
    tokenizer=xlnet_tokenizer,
)

# Perform predictions for XLNet with error handling
try:
    xlnet_preds = xlnet_trainer.predict(xlnet_ds).predictions
except Exception as e:
    print("Error during prediction:", str(e))
    raise  # Re-raise to see more details

# Apply softmax to predictions
xlnet_predictions = torch.softmax(torch.tensor(xlnet_preds), dim=-1)

# Add probabilities to the training dataframe
for i in range(xlnet_predictions.shape[1]):
    train_df[f'xlnet_prob_{i}'] = xlnet_predictions[:, i]

# Drop 'labels' column and show the updated dataframe
train_df.drop(columns=['labels'], inplace=True)

# Clean up resources for XLNet
del xlnet_model, xlnet_trainer
torch.cuda.empty_cache()
gc.collect()



In [ ]:
train_df.head()

In [ ]:
# Import necessary libraries
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
import torch
import gc

# Define GPT model path
GPT_MODEL_PATH = 'gpt4'  # Adjust to your desired GPT variant

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get the number of unique labels in the validation dataframe
unique_labels = set(val_df['labels'])  # Update as necessary
num_classes = len(unique_labels)

# Create configuration with the correct number of labels for GPT
gpt_config = AutoConfig.from_pretrained(GPT_MODEL_PATH, num_labels=num_classes)

# Initialize tokenizer and model for GPT with correct configuration
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_PATH)
gpt_model = AutoModelForSequenceClassification.from_config(gpt_config).to(device)

# Tokenize text data for GPT
def tokenize_gpt(sample):
    return gpt_tokenizer(sample['clean_text'], max_length=512, truncation=True)

# Load dataset for GPT
gpt_ds = Dataset.from_pandas(train_df).map(tokenize_gpt).remove_columns(['essay_id', 'clean_text'])

# Set training arguments for GPT
gpt_args = TrainingArguments(
    "gpt_predictions",
    per_device_eval_batch_size=10,
    report_to="none"
)

# Create a trainer for GPT
gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    data_collator=DataCollatorWithPadding(gpt_tokenizer),
    tokenizer=gpt_tokenizer,
)

# Perform predictions for GPT with error handling

try:
    gpt_preds = gpt_trainer.predict(gpt_ds).predictions
except Exception as e:
    print("Error during prediction:", str(e))
    raise  # Re-raise to see more details

# Apply softmax to GPT predictions if successful
gpt_predictions = torch.softmax(torch.tensor(gpt_preds), dim=-1)

# Add GPT probabilities to the validation dataframe
# Add probabilities to the training dataframe
for i in range(xlnet_predictions.shape[1]):
    train_df[f'gpt_prob_{i}'] = xlnet_predictions[:, i]

# Drop 'labels' column and show the updated dataframe

# train_df.drop(columns=['labels'], inplace=True)

# Clean up resources for GPT
del gpt_model, gpt_trainer
torch.cuda.empty_cache()
gc.collect()


In [ ]:

# Import necessary libraries
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
import torch
import gc

# Model and tokenizer paths for XLNet
XLNET_MODEL_PATH = 'xlnet-base-cased'  # Adjust to your desired XLNet variant

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get the number of unique labels in the dataset (assuming 'train_df' is defined)
unique_labels = set(train_df['labels'])  # Update as necessary
num_classes = len(unique_labels)

# Create configuration with the correct number of labels
xlnet_config = AutoConfig.from_pretrained(XLNET_MODEL_PATH, num_labels=num_classes)

# Initialize tokenizer and model with correct configuration
xlnet_tokenizer = AutoTokenizer.from_pretrained(XLNET_MODEL_PATH)
xlnet_model = AutoModelForSequenceClassification.from_config(xlnet_config).to(device)

# Tokenize text data for XLNet
def tokenize_xlnet(sample):
    return xlnet_tokenizer(sample['clean_text'], max_length=512, truncation=True)

# Load dataset for XLNet
xlnet_ds = Dataset.from_pandas(val_df).map(tokenize_xlnet).remove_columns(['essay_id', 'clean_text'])

# Set training arguments for XLNet
xlnet_args = TrainingArguments(
    "xlnet_predictions",
    per_device_eval_batch_size=10,
    report_to="none"
)

# Create a trainer for XLNet
xlnet_trainer = Trainer(
    model=xlnet_model,
    args=xlnet_args,
    data_collator=DataCollatorWithPadding(xlnet_tokenizer),
    tokenizer=xlnet_tokenizer,
)

# Perform predictions for XLNet with error handling
try:
    xlnet_preds = xlnet_trainer.predict(xlnet_ds).predictions
except Exception as e:
    print("Error during prediction:", str(e))
    raise  # Re-raise to see more details

# Apply softmax to predictions
xlnet_predictions = torch.softmax(torch.tensor(xlnet_preds), dim=-1)

# Add probabilities to the training dataframe
for i in range(xlnet_predictions.shape[1]):
    val_df[f'xlnet_prob_{i}'] = xlnet_predictions[:, i]

# Drop 'labels' column and show the updated dataframe
# val_df.drop(columns=['labels'], inplace=True)

# Clean up resources for XLNet
del xlnet_model, xlnet_trainer
torch.cuda.empty_cache()
gc.collect()



In [ ]:
# Import necessary libraries
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset
import torch
import gc

# Define GPT model path
GPT_MODEL_PATH = 'gpt4'  # Adjust to your desired GPT variant

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Get the number of unique labels in the validation dataframe
unique_labels = set(val_df['labels'])  # Update as necessary
num_classes = len(unique_labels)

# Create configuration with the correct number of labels for GPT
gpt_config = AutoConfig.from_pretrained(GPT_MODEL_PATH, num_labels=num_classes)

# Initialize tokenizer and model for GPT with correct configuration
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_MODEL_PATH)
gpt_model = AutoModelForSequenceClassification.from_config(gpt_config).to(device)

# Tokenize text data for GPT
def tokenize_gpt(sample):
    return gpt_tokenizer(sample['clean_text'], max_length=512, truncation=True)

# Load dataset for GPT
gpt_ds = Dataset.from_pandas(val_df).map(tokenize_gpt).remove_columns(['essay_id', 'clean_text'])

# Set training arguments for GPT
gpt_args = TrainingArguments(
    "gpt_predictions",
    per_device_eval_batch_size=10,
    report_to="none"
)

# Create a trainer for GPT
gpt_trainer = Trainer(
    model=gpt_model,
    args=gpt_args,
    data_collator=DataCollatorWithPadding(gpt_tokenizer),
    tokenizer=gpt_tokenizer,
)

# Perform predictions for GPT with error handling

try:
    gpt_preds = gpt_trainer.predict(gpt_ds).predictions
except Exception as e:
    print("Error during prediction:", str(e))
    raise  # Re-raise to see more details

# Apply softmax to GPT predictions if successful
gpt_predictions = torch.softmax(torch.tensor(gpt_preds), dim=-1)

# Add GPT probabilities to the validation dataframe
# Add probabilities to the training dataframe
for i in range(xlnet_predictions.shape[1]):
    val_df[f'gpt_prob_{i}'] = xlnet_predictions[:, i]

# Drop 'labels' column and show the updated dataframe

# train_df.drop(columns=['labels'], inplace=True)

# Clean up resources for GPT
del gpt_model, gpt_trainer
torch.cuda.empty_cache()
gc.collect()
